# HETERO-WORLD GPU jobs (Colab)
1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. When asked, allow Google Drive access and paste your Kaggle API token (hidden input; it is not stored in the notebook).
3. Results are written to `MyDrive/hetero_world_runs/` after every step, so a disconnect loses at most the run in progress. Re-running skips finished runs.

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU: set Runtime -> Change runtime type -> T4 GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/hetero_world_runs'
os.makedirs(OUT, exist_ok=True)
print('outputs ->', OUT)

In [ ]:
import getpass, os, subprocess
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
tok = getpass.getpass('Kaggle API token (KGAT_...): ').strip()
path = os.path.expanduser('~/.kaggle/access_token')
open(path, 'w').write(tok); os.chmod(path, 0o600); del tok
subprocess.run('pip -q install -U kaggle', shell=True, check=True)
for ds, dest in [('popsensharma/hetero-world-ethucy', '/content/ds_ethucy'), ('popsensharma/hetero-world-code', '/content/ds_code')]:
    if not os.path.isdir(dest):
        subprocess.run(f'kaggle datasets download {ds} -p {dest} --unzip', shell=True, check=True)
print(subprocess.run('ls -R /content/ds_ethucy | head -30', shell=True, capture_output=True, text=True).stdout)

In [ ]:
import glob, os, shutil, tarfile
WORK = '/content/hw'
if not os.path.isdir(WORK + '/hetero_world'):
    os.makedirs(WORK, exist_ok=True)
    arch = [p for p in glob.glob('/content/ds_ethucy/**/code.tar.gz', recursive=True) if os.path.isfile(p)]
    if arch:
        tarfile.open(arch[0]).extractall(WORK)
    else:
        for name in ('hetero_world', 'scripts'):
            hit = sorted(glob.glob(f'/content/ds_ethucy/**/{name}', recursive=True), key=len)[0]
            shutil.copytree(hit, f'{WORK}/{name}')
os.makedirs(WORK + '/data/ethucy', exist_ok=True)
for f in glob.glob('/content/ds_ethucy/**/*.pt', recursive=True):
    shutil.copy(f, WORK + '/data/ethucy/')
wm = glob.glob('/content/ds_code/**/wm_train.pt', recursive=True)[0]
if not os.path.exists(WORK + '/data/wm_train.pt'):
    os.symlink(wm, WORK + '/data/wm_train.pt')
os.chdir(WORK)
print(sorted(os.listdir('data/ethucy')), os.path.getsize('data/wm_train.pt') // 2**20, 'MB corridor data')

## 1. ETH/UCY real-pedestrian forecast study (leave-one-scene-out, 3 seeds)

In [ ]:
import subprocess, sys, os
res = f'{OUT}/ethucy_results.json'
if not os.path.exists(res):
    with open(f'{OUT}/ethucy.log', 'w') as log:
        p = subprocess.run([sys.executable, 'scripts/ethucy_experiment.py', '--seeds', '0', '1', '2', '--steps', '8000',
                            '--device', 'cuda', '--out', res], stdout=log, stderr=subprocess.STDOUT)
    print('exit', p.returncode)
print(open(f'{OUT}/ethucy.log').read()[-5000:])

## 2. Learned world models on the corridor (diffusion / JEPA / RSSM x seeds 0-2, fixed step budgets)

In [ ]:
import subprocess, sys, os, shutil, time
STEPS = {'diffusion': 60000, 'jepa': 40000, 'rssm': 30000}
os.makedirs(f'{OUT}/logs', exist_ok=True)
for seed in (0, 1, 2):
    for fam, steps in STEPS.items():
        ck_dir = 'checkpoints' if seed == 0 else f'checkpoints/seed{seed}'
        if os.path.exists(f'{OUT}/{ck_dir}/wm_{fam}.pt'):
            print('skip', fam, seed); continue
        t0 = time.time()
        log = f'{OUT}/logs/train_{fam}_seed{seed}.log'
        with open(log, 'w') as f:
            p = subprocess.run([sys.executable, 'scripts/train_world_model.py', '--family', fam, '--seed', str(seed),
                                '--max-steps', str(steps), '--minutes', '600'], stdout=f, stderr=subprocess.STDOUT)
        print(fam, 'seed', seed, 'exit', p.returncode, f'{time.time() - t0:.0f}s', open(log).read().strip().splitlines()[-1][:200])
        shutil.copytree('checkpoints', f'{OUT}/checkpoints', dirs_exist_ok=True)
print('all done')